# Linear Models — Full Automated Calibration Dataset

## Decision

The requested models were fitted on **all 162 automated sessions** and **149,858 valid synchronized frames**. They are reproducible, but the held-out-session results say they are **not reliable enough for deployment**:

- Force: MAE about **0.53 N**, RMSE about **0.81 N**, and R² about **0.02**.
- Nine-ROI localization: **45.0%** contact-frame accuracy and **54.9%** session-level accuracy (chance is 11.1%).

The artifact is therefore useful as a measured baseline and implementation scaffold, not as a foolproof sensor model.


## Method

- Inputs: ROI-area-normalized `delta_v_mean` and `active_fraction` for ROIs 1–9.
- Camera handling: one camera-configuration indicator plus camera-by-feature interactions, because the intentional camera change altered the optical scale while the skin stayed the same.
- Force: ridge-regularized linear regression on every valid frame.
- Localization: nine one-vs-rest linear score regressions on contact frames (`force_N >= 0.05`), with the largest score selected.
- Validation: five grouped folds; a complete acquisition session is always entirely train or test. Training weights give every session equal total weight.
- Integrated optical response is not divided by cycle count for these instantaneous models. Mean/per-pixel ROI response is the appropriate area normalization. Force-time normalization would only be appropriate for a separate session-level energy/impulse question.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'linear_models').exists():
    ROOT = Path(r'C:\Users\DLSU\OneDrive\Documents\SOFTWARE CALIBRATION\analysis_outputs')
MODEL_DIR = ROOT / 'linear_models'
metrics = json.loads((MODEL_DIR / 'validation_metrics.json').read_text())
model = json.loads((MODEL_DIR / 'auto_calibration_linear_models.json').read_text())
sessions = pd.read_csv(MODEL_DIR / 'session_level_oof_predictions.csv')
confusion = pd.read_csv(MODEL_DIR / 'localization_confusion_frames.csv', index_col=0)
model['training_scope']


{'archive': 'C:\\Users\\DLSU\\Downloads\\Data Collection\\Auto Calibration.zip', 'sessions': 162, 'valid_frames': 149858, 'contact_frames': 114290, 'same_physical_skin': True, 'camera_change_intentional': True}


## Cross-validated results

In [2]:
pd.DataFrame({
    'all valid frames': metrics['force_all_frames'],
    'contact frames only': metrics['force_contact_frames'],
})


                                   all valid frames  contact frames only
n_frames                              149858.000000        114290.000000
mae_N                                      0.526155             0.526970
rmse_N                                     0.810174             0.875867
r2                                         0.016534            -0.043111
bias_N                                    -0.002168            -0.165754
noncontact_predicted_contact_rate          0.995839                  NaN
contact_detection_sensitivity              0.997541                  NaN


In [3]:
pd.Series({
    'contact-frame accuracy': metrics['localization_contact_frame_accuracy'],
    'session-level accuracy': metrics['localization_session_accuracy'],
    'correct sessions': metrics['localization_session_correct'],
    'total sessions': metrics['localization_sessions'],
    'chance accuracy': 1/9,
})


contact-frame accuracy      0.449707
session-level accuracy      0.549383
correct sessions           89.000000
total sessions            162.000000
chance accuracy             0.111111
dtype: float64


In [4]:
pd.DataFrame(metrics['localization_per_roi']).set_index('target_roi')


            sessions  session_accuracy
target_roi                            
1                 18          1.000000
2                 18          0.611111
3                 18          0.111111
4                 18          0.500000
5                 18          0.444444
6                 18          0.333333
7                 18          0.500000
8                 18          1.000000
9                 18          0.444444


In [5]:
pd.DataFrame(metrics['localization_per_camera']).set_index('camera_configuration')


                      sessions  session_accuracy
camera_configuration                            
A                           99          0.444444
B                           63          0.714286


### Interpretation

The force model is barely better than predicting an overall mean (`R² ≈ 0.02`) and incorrectly crosses the 0.05 N contact threshold for nearly every true non-contact frame. It must not be used as a contact detector.

Localization is above chance, showing that the full nine-ROI pattern contains information, but the error is strongly location-dependent: ROI 3 is only 11.1% correct at session level, while ROIs 1 and 8 are 100% in this cross-validation. Camera configuration A is also substantially worse than B. This unevenness is exactly why a simple “brightest ROI pops up” rule is not foolproof.


## Saved model and scoring equation

In [6]:
def linear_predict(saved, engineered_features):
    x = np.asarray(engineered_features, dtype=float)
    mean = np.asarray(saved['feature_mean'])
    scale = np.asarray(saved['feature_scale'])
    coef = np.asarray(saved['standardized_coefficients'])
    intercept = np.asarray(saved['target_mean_or_intercepts'])
    return (x - mean) / scale @ coef + intercept

{
    'force_features': len(model['force_model']['feature_names']),
    'localization_features': len(model['localization_model']['feature_names']),
    'camera_configurations': model['camera_configurations'],
    'localization_classes': model['localization_model']['classes'],
}


{'force_features': 37, 'localization_features': 55, 'camera_configurations': {'A': '5a40306c002688d3274d10fb1dd05f03f54dfee4b3201d22c8cd7cf21dca0478', 'B': '4cde17769ad834a7ff4b578d714cb13c8e8f28b980762587caadaaa8e6808b25'}, 'localization_classes': [1, 2, 3, 4, 5, 6, 7, 8, 9]}


For force, clamp the scalar result to zero. For localization, average the nine linear scores over five consecutive frames that are known to be in contact, then choose the largest score. The JSON stores feature order, means, scales, coefficients, camera fingerprints, and class order.

Important: the localization feature vector contains raw ROI features, camera interactions, and positive per-frame ROI shares. `build_linear_models.py` is the reference feature-engineering implementation; using a different order or normalization invalidates the coefficients.


## Validation checks

In [7]:
checks = {
    '162 unique held-out sessions': sessions.session.nunique() == 162,
    'one fold per session': sessions.groupby('session').fold.nunique().max() == 1,
    'all nine actual ROIs present': set(sessions.target_roi) == set(range(1, 10)),
    'confusion total equals contact frames': int(confusion.to_numpy().sum()) == model['training_scope']['contact_frames'],
    'two known camera configurations': len(model['camera_configurations']) == 2,
    'finite force coefficients': bool(np.isfinite(model['force_model']['standardized_coefficients']).all()),
    'finite localization coefficients': bool(np.isfinite(model['localization_model']['standardized_coefficients']).all()),
}
assert all(checks.values())
checks


{'162 unique held-out sessions': True, 'one fold per session': np.True_, 'all nine actual ROIs present': True, 'confusion total equals contact frames': True, 'two known camera configurations': True, 'finite force coefficients': True, 'finite localization coefficients': True}


## Practical next step

Because a fixed-force repeat at every ROI is unavailable, the most robust achievable localization approach is a **manual per-ROI template calibration** on the same skin and frozen camera settings: press each ROI a few times, store its nine-ROI response pattern, and match new contact patterns to those templates. Keep the load cell for force until a genuinely predictive optical force model validates on untouched sessions.

The current linear models can still be integrated behind a clearly marked experimental flag. Do not allow them to control safety limits or report force as a calibrated measurement.
